# BMKMANU S1000 - exp74 100M

## Import packages

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import harpy as hp
import spatialdata as sd
import pandas as pd
import numpy as np
import os
import geopandas as gpd
import matplotlib
from matplotlib import pyplot as plt
import gc
import spatialdata_plot
from spatialdata.transformations import get_transformation, remove_transformation, set_transformation

matplotlib.use("Agg")

import logging
logging.getLogger('ome_zarr').setLevel(logging.ERROR) # This mutes unnecessary Zarr warning (see: https://github.com/scverse/spatialdata/issues/400)

/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/scripts/2024-07-UTBenchmark-SpC/.venv_harpy_20Jan2026/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [3]:
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))
from utils.in_tissue import add_in_tissue
from utils.plotting import best_datatree_scale, rel_fontsize
from utils.qc import qc_metrics
from _bmkmanu import bmkmanu

## Create sdata

In [4]:
dataset_id = 'exp74'

path = f'/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/{dataset_id}/subsampled_100M/BSTMAtrix'
bc_path = f'/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/{dataset_id}/bc2bc/{dataset_id}_100M.bc'
config_prefix = 'exp74_100M_11Aug2025'
technology = 'S1000'
output_path = f'/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/{dataset_id}/subsampled_100M/harpy/sdata.zarr'

barcode_decoding_path = os.path.join(f"/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/raw/BMK_S1000/{dataset_id}", "20231101-CH11BN17F4-A3-20X-V2AB-final_signal.txt")

pipeline_name = 'BSTMAtrix'
species = 'mouse'
tissue_annotation_file = f'/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/{dataset_id}/tissue_annotation/tissue.geojson'


In [5]:
# sdata = sd.read_zarr(output_path)
# sdata

In [6]:
sdata = bmkmanu(
    path=path, 
    dataset_id=dataset_id,
    filtered_counts=False,
    add_in_tissue_column=True,
    levels=None, 
    technology=technology,
    bc_path=bc_path,
    microns_per_pixel = 0.341, # Based on calculations below
    output=output_path
)
sdata

2026-01-28 14:50:52.391 | WARNING  | harpy.io._spatial_data:_load_image_to_dask:231 - dims parameter is equal to ['c', 'y', 'x'], but will be ignored when reading in images from a file
2026-01-28 14:51:04.649 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'exp74_he'
2026-01-28 14:51:20.555 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'exp74_level_5_labels'
2026-01-28 14:52:05.784 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'exp74_level_2_labels'
2026-01-28 14:52:19.168 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'exp74_level_6_labels'
2026-01-28 14:52:40.110 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'exp74_level_3_labels'
2026-01-28 14:52:46.046 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'exp74_level_13_labels'
2026-01-28 14:52:58.088 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'e

SpatialData object, with associated Zarr store: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/sdata.zarr
├── Images
│     └── 'exp74_he': DataTree[cyx] (3, 20000, 19886), (3, 10000, 9943), (3, 5000, 4971), (3, 2500, 2485), (3, 1250, 1242)
├── Labels
│     ├── 'exp74_level_1_labels': DataArray[yx] (19983, 19872)
│     ├── 'exp74_level_2_labels': DataArray[yx] (20036, 19929)
│     ├── 'exp74_level_3_labels': DataArray[yx] (20083, 19980)
│     ├── 'exp74_level_4_labels': DataArray[yx] (20131, 20032)
│     ├── 'exp74_level_5_labels': DataArray[yx] (20179, 20083)
│     ├── 'exp74_level_6_labels': DataArray[yx] (20226, 20135)
│     ├── 'exp74_level_7_labels': DataArray[yx] (20274, 20186)
│     └── 'exp74_level_13_labels': DataArray[yx] (20561, 20496)
├── Shapes
│     ├── 'exp74_level_1_shapes': GeoDataFrame shape: (1844233, 1) (2D shapes)
│     ├── 'exp74_level_2_shapes': GeoDataFrame shape: (337790, 1) (2D sh

In [7]:
levels = [level.split(f'{dataset_id}_')[1].split("_table")[0] for level in [*sdata.tables]]
print(f"{pipeline_name} bins: {levels}")

BSTMAtrix bins: ['level_13', 'level_1', 'level_2', 'level_3', 'level_4', 'level_5', 'level_6', 'level_7']


In [8]:
# Assert whether number of shapes matches number of label IDs
import dask.array as da

for level in levels:
    number_of_shapes = len(sdata.shapes[f"{dataset_id}_{level}_shapes"])
    number_of_label_IDs = len(da.unique(sdata.labels[f"{dataset_id}_{level}_labels"].data).compute()) - 1
    assert number_of_shapes == number_of_label_IDs, f"{level}: Number of shapes ({number_of_shapes}) does not match number of label IDs ({number_of_label_IDs})"

## Calculate pixel size

In [9]:
# Determine pixel size in microns
he_shape = hp.im._image._get_spatial_element(sdata, f'{dataset_id}_he').shape

he_width = he_shape[2]
he_height = he_shape[1]

microns_per_pixel_x = 6800 / he_shape[2]
microns_per_pixel_y = 6800 / he_shape[1]
print(f"Pixel size: {microns_per_pixel_x:.3f} µm (X), {microns_per_pixel_y:.3f} µm (Y)")

microns_per_pixel = (microns_per_pixel_x + microns_per_pixel_y) / 2
print(f"Pixel size: {microns_per_pixel:.3f} µm")

# NOTE: This is not a great way to have to surmise what the pixel size is. This was recommended by BMKGENE and should be good enough, but there is definitely the possibility that the capture area is not exactly 6.8 mm.

Pixel size: 0.342 µm (X), 0.340 µm (Y)
Pixel size: 0.341 µm


## Hexagonal tesselation

In [10]:
custom_bins = ['hexagonal_grid_20um', 'hexagonal_grid_120um'] # Names refer to the diagonal of the hexagon

In [11]:
min_x, min_y, max_x, max_y = sdata.shapes[f"{dataset_id}_level_1_shapes"]['geometry'].total_bounds
shape = (max_y - min_y, max_x - min_x)

for custom_bin in custom_bins:
    gc.collect()
    
    size = (int(custom_bin.split("hexagonal_grid_")[1].split("um")[0])/2) / microns_per_pixel # Radius of the hexagon in pixels
    
    # Create hexagonal grids
    sdata = hp.im.add_grid_labels_layer(
        sdata, 
        shape=shape,
        size=size,
        output_shapes_layer=f"{custom_bin}_shapes", 
        output_labels_layer=f"{custom_bin}_labels", 
        grid_type="hexagon", 
        offset=(0, 0), 
        chunks=2048, 
        client=None, 
        transformations=get_transformation(sdata.shapes[f"{dataset_id}_level_1_shapes"], get_all=True),
        scale_factors=(2, 2, 2, 2),
        overwrite=True
    )

    # Rebin data based on hexagonal grid
    sdata = hp.tb.bin_counts( 
        sdata,
        table_layer=f"{dataset_id}_level_1_table",
        labels_layer=f"{custom_bin}_labels",
        output_layer=f"{custom_bin}_table",
        append = False,
        to_coordinate_system = dataset_id,
        overwrite=True,
    )

2026-01-28 15:00:09.125 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'hexagonal_grid_20um_labels'
2026-01-28 15:00:24.098 | INFO     | harpy.table._allocation:_aggregate:488 - Calculating cell counts.
2026-01-28 15:00:28.533 | INFO     | harpy.table._allocation:bin_counts:363 - 
 Number of spots per bin  Frequency
                       1         73
                       2        330
                       3        951
                       4       2180
                       5       4353
                       6       7081
                       7      10560
                       8      14685
                       9      17443
                      10      21939
                      11      28926
                      12      31410
                      13      22835
                      14      10763
                      15       3359
                      16          1
2026-01-28 15:01:02.302 | INFO     | harpy.image._manager:add_layer:47 - Writin

## Specify plotting parameters

In [12]:
DPI = 300
MAX_DIM_FOR_PLOTS = 10000

HE_image = f'{dataset_id}_he'

best_scale = best_datatree_scale(sdata[HE_image], MAX_DIM_FOR_PLOTS)
print(f'best scale: {best_scale}')

figsize_factor = DPI
figsize = (sdata.images[HE_image][best_scale]['image'].shape[2]/figsize_factor, sdata.images[HE_image][best_scale]['image'].shape[1]/figsize_factor)
print(f'figsize: {figsize}')

plots_path = os.path.join(os.path.dirname(output_path), 'plots')
os.makedirs(plots_path, exist_ok=True)
print(f'plots_path: {plots_path}')

best scale: scale1
figsize: (33.14333333333333, 33.333333333333336)
plots_path: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots


## Expand labels layers for plotting

In [13]:
# Expand labels layer masks for plotting (otherwise, the plots are not very clear)
for level in ['level_1']:
    
    radius = np.sqrt(sdata.shapes[f"{dataset_id}_{level}_shapes"].geometry.iloc[0].area / np.pi)
    
    sdata = hp.im.expand_labels_layer(
        sdata,
        labels_layer=f"{dataset_id}_{level}_labels",
        distance=(radius * 1.5) - radius, # Radius increased by 150%
        output_labels_layer=f"{dataset_id}_{level}_labels",
        overwrite=True,
)
    
# NOTE: The result of this is that plots using the labels layer do not show spots to scale, but the visualization is improved.

2026-01-28 15:01:34.249 | INFO     | harpy.image.segmentation._map:_combine_dask_arrays:339 - Linking labels across chunks.
2026-01-28 15:01:34.374 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'exp74_level_1_labels'
2026-01-28 15:01:39.365 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'exp74_level_1_labels' already exists. Overwriting...


## Create crop for zoomed-in plots

In [14]:
min_x = sdata.images[HE_image]['scale0']['image'].shape[2] / 2 # Center of image
max_x = min_x + round(200 / microns_per_pixel) # 200 micrometer
min_y = sdata.images[HE_image]['scale0']['image'].shape[1] / 2 # Center of image
max_y = min_y + round(200 / microns_per_pixel) # 200 micrometer

sdata_crop = sdata.query.bounding_box(
    min_coordinate=[min_x, min_y], 
    max_coordinate=[max_x, max_y], 
    axes=("x", "y"), 
    target_coordinate_system=dataset_id
)

figsize_crop = (3,3)

## Plot H&E

In [15]:
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "H&E"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, 'HE.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/HE.png


## Plot BSTMAtrix spots

In [16]:
for level in levels:
    gc.collect()

    level_plots_path = os.path.join(plots_path, level)
    os.makedirs(level_plots_path, exist_ok=True)

    # Plot full image
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{pipeline_name} - spots"

    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_shapes(
            element=f"{dataset_id}_{level}_shapes", fill_alpha=0.15, outline_width=1, outline_alpha=1, color='#41b8de', outline_color='#41b8de'
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, f'{pipeline_name}_spots.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

    # Plot crop
    fig, ax = plt.subplots(figsize=figsize_crop, dpi=DPI)

    title = f"{pipeline_name} - spots"

    sdata_crop.pl.render_images(
        element=HE_image, scale="full"
    ).pl.render_shapes(
            element=f"{dataset_id}_{level}_shapes", fill_alpha=0.15, outline_width=0.25, outline_alpha=1, color='#41b8de', outline_color='#41b8de', method="matplotlib"
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize_crop,
        colorbar=False,
        dpi=DPI,
        ax=ax,
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_y, max_y)
    ax.invert_yaxis()

    save_path = os.path.join(level_plots_path, f'crop_{pipeline_name}_spots.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)
    
# NOTE: Level 1 spots are represented true to scale. Other levels have radii that maximize visualization.

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/level_13/BSTMAtrix_spots.png
INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/level_13/crop_BSTMAtrix_spots.png
INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                      

## Plot hexagonal spots

In [17]:
for custom_bin in custom_bins:
    gc.collect()

    level_plots_path = os.path.join(plots_path, custom_bin)
    os.makedirs(level_plots_path, exist_ok=True)

    # Plot full image
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"Custom grid - {custom_bin}"
    
    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_shapes(
            element=f"{custom_bin}_shapes", fill_alpha=0.15, outline_width=1, outline_alpha=1, color='#41b8de', outline_color='#41b8de'
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, f'{custom_bin}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

    # Plot crop
    fig, ax = plt.subplots(figsize=figsize_crop, dpi=DPI)

    title = f"Custom grid - {custom_bin}"
    
    sdata_crop.pl.render_images(
        element=HE_image, scale="full"
    ).pl.render_shapes(
            element=f"{custom_bin}_shapes", fill_alpha=0.15, outline_width=0.25, outline_alpha=1, color='#41b8de', outline_color='#41b8de', method="matplotlib"
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_y, max_y)
    ax.invert_yaxis()

    save_path = os.path.join(level_plots_path, f'crop_{custom_bin}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)
    
# NOTE: Spots are represented true to scale

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/hexagonal_grid_20um/hexagonal_grid_20um.png
INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/hexagonal_grid_20um/crop_hexagonal_grid_20um.png
INFO     Value for paramete

## Plot BSTMAtrix tissue annotation

In [18]:
for level in levels:
    gc.collect()
    
    level_plots_path = os.path.join(plots_path, level)
    os.makedirs(level_plots_path, exist_ok=True)

    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{pipeline_name} - {pipeline_name} tissue annotation"
    
    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_labels(
        element=f"{dataset_id}_{level}_labels", fill_alpha=0.4, contour_px=5, outline_alpha=0, color='in_tissue', table_name=f"{dataset_id}_{level}_table", scale=best_scale
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, f'{pipeline_name}_tissue_annotation.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/level_13/BSTMAtrix_tissue_annotation.png
INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/level_1/BSTMAtrix_tissue_annotation.png
INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/level_2/BSTMAtrix_tissue_annotation.png
INFO     Rasterizing image for faster rendering.                                 

## Create bounding box for total capture area

In [19]:
# Add bounding box for total capture area
import shapely

geoms = sdata.shapes[f'{dataset_id}_level_1_shapes'].geometry
bbox = shapely.MultiPolygon(list(geoms)).minimum_rotated_rectangle

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gpd.GeoDataFrame(geometry=[bbox]),
    output_layer = f'{dataset_id}_capture_area',
    transformations = get_transformation(sdata.shapes[f'{dataset_id}_level_1_shapes'], get_all=True),
    overwrite = True,
)

In [21]:
# Plot capture area bbox
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Capture area bounding box"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_shapes(
    element=f'{dataset_id}_capture_area', color='red', fill_alpha=0.05, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.05)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, 'capture_area_bbox.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/capture_area_bbox.png


In [22]:
# Plot capture area bbox
for level in ["level_1"]:
    gc.collect()
    
    level_plots_path = os.path.join(plots_path, level)
    os.makedirs(level_plots_path, exist_ok=True)

    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{pipeline_name} - Capture area bounding box"
    
    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_shapes(
        element=f'{dataset_id}_capture_area', color='red', fill_alpha=0.05, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
    ).pl.render_shapes(
        element=f"{dataset_id}_{level}_shapes", fill_alpha=0, outline_width=1, outline_alpha=1, outline_color='#41b8de'
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, 'capture_area_bbox.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/level_1/capture_area_bbox.png


In [23]:
# Area of capture area in mm²
gdf = sdata.shapes[f'{dataset_id}_capture_area']
capture_area = (gdf.geometry.area.sum() * microns_per_pixel**2) / 1000000
print(f"Area of capture area: {capture_area:.2f} mm²")

# NOTE: This is biased because we are using the size of the capture area to determine the pixel size. However, it should be close.

Area of capture area: 46.17 mm²


In [24]:
from math import dist
coords = list(bbox.exterior.coords)

# Compute lengths of bbox
side1 = dist(coords[0], coords[1]) * microns_per_pixel / 10000
side2 = dist(coords[1], coords[2]) * microns_per_pixel / 10000

print(f"Dimensions of capture area bbox: {side1:.2f} cm x {side2:.2f} cm")

# NOTE: This is biased because we are using the size of the capture area to determine the pixel size

Dimensions of capture area bbox: 0.68 cm x 0.68 cm


## Add tissue annotation

In [25]:
# Add tissue annotation as shapes layer and as labels layer
gdf = gpd.read_file(tissue_annotation_file)
gdf = gdf.explode(index_parts=False) # This avoid problems with MultiPolygon geometries
gdf.index = gdf.index + 1 # Index 0 is not allowed

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gdf,
    output_layer = f'{dataset_id}_tissue',
    transformations = get_transformation(sdata.images[f'{dataset_id}_he'], get_all=True),
    overwrite = True,
)

sdata = hp.im.rasterize(
    sdata,
    shapes_layer = f'{dataset_id}_tissue',
    output_layer = f'{dataset_id}_tissue_mask',
    chunks = 1024,
    overwrite = True,
)

2026-01-28 15:33:10.677 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'exp74_tissue_mask'


In [26]:
# Check tissue annotation
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Tissue annotation"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_shapes(
    element=f'{dataset_id}_tissue', color='#1f77b4', fill_alpha=0.3, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, 'Qupath_tissue_annotation.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/Qupath_tissue_annotation.png


In [27]:
# Determine which spots are in tissue annotation
for level in levels:
    gc.collect()
    sdata = add_in_tissue(
        sdata, 
        table_layer = f"{dataset_id}_{level}_table", 
        tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
    )

for custom_bin in custom_bins:
    gc.collect()
    sdata = add_in_tissue(
        sdata, 
        table_layer = f"{custom_bin}_table", 
        tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
    )

2026-01-28 15:34:23.781 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'exp74_level_13_table' already exists. Overwriting...
2026-01-28 15:35:05.874 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'exp74_level_1_table' already exists. Overwriting...
2026-01-28 15:35:36.445 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'exp74_level_2_table' already exists. Overwriting...
2026-01-28 15:36:00.548 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'exp74_level_3_table' already exists. Overwriting...
2026-01-28 15:36:23.264 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'exp74_level_4_table' already exists. Overwriting...
2026-01-28 15:36:45.493 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'exp74_level_5_table' already exists. Overwriting...
2026-01-28 15:37:08.121 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with 

In [28]:
# Check spots in tissue
for level in levels:
    gc.collect()

    level_plots_path = os.path.join(plots_path, level)
    os.makedirs(level_plots_path, exist_ok=True)

    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{level} - Qupath tissue annotation"

    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_labels(
        element=f"{dataset_id}_{level}_labels", fill_alpha=0.4, contour_px=5, outline_alpha=0, color='in_tissue_annotation', table_name=f"{dataset_id}_{level}_table", scale=best_scale
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, 'Qupath_tissue_annotation_spots.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/level_13/Qupath_tissue_annotation_spots.png
INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/level_1/Qupath_tissue_annotation_spots.png
INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/level_2/Qupath_tissue_annotation_spots.png
INFO     Rasterizing image for faster rendering.                        

In [29]:
# Check spots in tissue
for custom_bin in custom_bins:
    gc.collect()

    level_plots_path = os.path.join(plots_path, custom_bin)
    os.makedirs(level_plots_path, exist_ok=True)

    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{custom_bin} - Qupath tissue annotation"

    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_labels(
        element=f"{custom_bin}_labels", fill_alpha=0.4, contour_px=5, outline_alpha=0, color='in_tissue_annotation', table_name=f"{custom_bin}_table", scale=best_scale
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, 'Qupath_tissue_annotation_spots.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/hexagonal_grid_20um/Qupath_tissue_annotation_spots.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/hexagonal_grid_120um/Qupath_tissue_annotation_spots.png


## QC

In [30]:
# Calculate QC metrics using scanpy's calculate_qc_metrics()
for level in levels:
    gc.collect()
    sdata = qc_metrics(
        sdata = sdata,
        table_layer = f"{dataset_id}_{level}_table",
        species = species
    )

for custom_bin in custom_bins:
    gc.collect()
    sdata = qc_metrics(
        sdata = sdata,
        table_layer = f"{custom_bin}_table",
        species = species
    )

2026-01-28 15:53:34.244 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'exp74_level_13_table' already exists. Overwriting...
2026-01-28 15:54:08.138 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'exp74_level_1_table' already exists. Overwriting...
2026-01-28 15:54:41.271 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'exp74_level_2_table' already exists. Overwriting...
2026-01-28 15:55:08.404 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'exp74_level_3_table' already exists. Overwriting...
2026-01-28 15:55:35.070 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'exp74_level_4_table' already exists. Overwriting...
2026-01-28 15:56:01.445 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'exp74_level_5_table' already exists. Overwriting...
2026-01-28 15:56:27.983 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with 

In [31]:
# Plot QC metrics
for level in levels:
    for qc_metric in ['n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'total_counts_mt', 'pct_counts_mt']:
        gc.collect()

        level_plots_path = os.path.join(plots_path, level)
        os.makedirs(level_plots_path, exist_ok=True)

        values = sdata.tables[f"{dataset_id}_{level}_table"].obs[qc_metric].to_numpy()
        vmin, vmax = np.percentile(values, [1, 99])
        norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
        
        fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

        title = f"Scanpy QC - {qc_metric}"

        sdata.pl.render_images(
            element=HE_image, scale=best_scale
        ).pl.render_labels(
            element=f"{dataset_id}_{level}_labels", fill_alpha=1, contour_px=5, outline_alpha=0, color=qc_metric, scale=best_scale, table_name=f"{dataset_id}_{level}_table", cmap="viridis", norm=norm
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=True,
            dpi=DPI,
            ax=ax,
            pad_extent=rel_fontsize(fig, 0.05)
        )

        ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
        ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
        ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
        ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

        cbar = fig.axes[-1]
        cbar.set_ylabel(qc_metric, rotation=270, labelpad=rel_fontsize(fig, 0.025), fontsize = rel_fontsize(fig, 0.02))
        cbar.tick_params(axis="y", which="both", labelsize=rel_fontsize(fig, 0.015))

        save_path = os.path.join(level_plots_path, f'Scanpy_QC_{qc_metric}.png')
        fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path, flush=True)

INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/level_13/Scanpy_QC_n_genes_by_counts.png
INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/level_13/Scanpy_QC_total_counts.png
INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/level_13/Scanpy_QC_pct_counts_in_top_50_genes.png
INFO     Rasterizing image for faster rendering.                           

In [32]:
# Plot QC metrics
for custom_bin in custom_bins:
    for qc_metric in ['n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'total_counts_mt', 'pct_counts_mt']:
        gc.collect()

        level_plots_path = os.path.join(plots_path, custom_bin)
        os.makedirs(level_plots_path, exist_ok=True)

        values = sdata.tables[f"{custom_bin}_table"].obs[qc_metric].to_numpy()
        vmin, vmax = np.percentile(values, [1, 99])
        norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
        
        fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

        title = f"Scanpy QC - {qc_metric}"

        sdata.pl.render_images(
            element=HE_image, scale=best_scale
        ).pl.render_labels(
            element=f"{custom_bin}_labels", fill_alpha=1, contour_px=5, outline_alpha=0, color=qc_metric, scale=best_scale, table_name=f"{custom_bin}_table", cmap="viridis", norm=norm
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=True,
            dpi=DPI,
            ax=ax,
            pad_extent=rel_fontsize(fig, 0.05)
        )

        ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
        ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
        ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
        ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

        cbar = fig.axes[-1]
        cbar.set_ylabel(qc_metric, rotation=270, labelpad=rel_fontsize(fig, 0.025), fontsize = rel_fontsize(fig, 0.02))
        cbar.tick_params(axis="y", which="both", labelsize=rel_fontsize(fig, 0.015))

        save_path = os.path.join(level_plots_path, f'Scanpy_QC_{qc_metric}.png')
        fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path, flush=True)

Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/hexagonal_grid_20um/Scanpy_QC_n_genes_by_counts.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/hexagonal_grid_20um/Scanpy_QC_total_counts.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/hexagonal_grid_20um/Scanpy_QC_pct_counts_in_top_50_genes.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/hexagonal_grid_20um/Scanpy_QC_total_counts_mt.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/hexagonal_grid_20um/Scanpy_QC_pct_counts

## Leiden clustering

In [33]:
from harpy.utils._keys import _INSTANCE_KEY, _REGION_KEY

for custom_bin in custom_bins:
    gc.collect()
    
    print(f'running: {custom_bin}', flush=True)

    labels_layer=f"{custom_bin}_labels"
    shapes_layer=f"{custom_bin}_shapes"
    table_layer=f"{custom_bin}_table"
    
    level_plots_path = os.path.join(plots_path, custom_bin)
    os.makedirs(level_plots_path, exist_ok=True)
    
    # Only keep spots in tissue for clustering
    adata = sdata.tables[table_layer][sdata.tables[table_layer].obs['in_tissue_annotation'] == True]
    
    sdata = hp.tb.add_table_layer(
        sdata,
        adata=adata.copy(),
        output_layer=f"{table_layer}_processed",
        region=adata.obs[_REGION_KEY].cat.categories.to_list(),
        overwrite=True,
    )
    
    # Preprocess data
    sdata = hp.tb.preprocess_transcriptomics(
        sdata=sdata,
        size_norm=False,
        labels_layer=labels_layer,
        table_layer=f"{table_layer}_processed",
        output_layer=f"{table_layer}_processed",
        min_counts=10,
        min_cells=5,
        highly_variable_genes=True,
        max_value_scale=10,
        n_comps=50,
        update_shapes_layers=False,
        overwrite=True
    )
    
    gc.collect()
    
    # Leiden clustering
    leiden_resolutions = [0.75, 1, 1.25]
    calculate_umap = True
    calculate_neighbors = True

    for res in leiden_resolutions:
        gc.collect()

        res_str = str(res).replace('.', '')
        print(f'Leiden resolution: {res}', flush=True)
                
        sdata = hp.tb.leiden(
            sdata, 
            labels_layer=labels_layer,
            table_layer=f"{table_layer}_processed",
            output_layer=f"{table_layer}_processed",
            calculate_umap=calculate_umap,
            calculate_neighbors=calculate_neighbors,
            rank_genes=False,
            n_neighbors=35,
            n_pcs=30,
            resolution=res,
            key_added=f'leiden_{res_str}',
            index_names_var=None,
            index_positions_var=None,
            random_state=100,
            overwrite=True
        )
        
        calculate_umap = False # Only calculate UMAP once
        calculate_neighbors = False # Only calculate neighbors once

        # Plot Leiden clustering
        gc.collect()
        
        with matplotlib.rc_context({'legend.markerscale': 2}):
            fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

            title = f"Leiden {res}"

            sdata.pl.render_images(
                element=HE_image, scale=best_scale
            ).pl.render_labels(
                element=labels_layer, fill_alpha=1, contour_px=5, outline_alpha=0, cmap="rainbow", color=f'leiden_{res_str}', table_name=f"{table_layer}_processed", scale=best_scale
            ).pl.show(
                coordinate_systems=dataset_id,
                title=title,
                figsize=figsize,
                colorbar=False,
                legend_fontsize=rel_fontsize(fig, 0.015), 
                legend_fontweight='bold', 
                legend_loc='right margin',
                na_in_legend=False,
                dpi=DPI,
                ax=ax,
            )

            ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
            ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
            ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
            ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

            save_path = os.path.join(level_plots_path, f'Leiden_{res}.png')
            fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

            plt.close(fig)

            print("Plot saved at:", save_path, flush=True)
        
        hp.pl.cluster(
            sdata, 
            table_layer = f"{table_layer}_processed",
            key_added=f'leiden_{res_str}',
            output=os.path.join(level_plots_path, f'Leiden_{res}')
        )

running: hexagonal_grid_20um


2026-01-28 16:49:07.028 | INFO     | harpy.table._preprocess:preprocess:314 - Calculating cell size from provided labels_layer 'hexagonal_grid_20um_labels'
2026-01-28 16:49:11.074 | INFO     | harpy.table._preprocess:preprocess:334 - Saving non preprocessed data matrix to the AnnData '.layers[raw_counts]'.
2026-01-28 16:49:44.688 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_20um_table_processed' already exists. Overwriting...


Leiden resolution: 0.75


2026-01-28 16:55:55.171 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_20um_table_processed' already exists. Overwriting...


Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/hexagonal_grid_20um/Leiden_0.75.png
Leiden resolution: 1


...

Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/hexagonal_grid_120um/Leiden_0.75.png
Leiden resolution: 1


2026-01-28 17:12:29.926 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_120um_table_processed' already exists. Overwriting...


Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/hexagonal_grid_120um/Leiden_1.png
Leiden resolution: 1.25


2026-01-28 17:14:19.488 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_120um_table_processed' already exists. Overwriting...


Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/plots/hexagonal_grid_120um/Leiden_1.25.png


## Spots metrics

In [34]:
gc.collect()

15141

In [35]:
with open(barcode_decoding_path, "r") as f:
    line_count = sum(1 for _ in f)
    unique_lines = set(line.strip() for line in f)

print(f"Lines in barcode decoding file: {line_count:,}")

Lines in barcode decoding file: 2,221,800


In [36]:
barcode_pos = pd.read_csv(os.path.join(path, config_prefix, 'barcode_pos.tsv'), sep='\t', header=None)
barcode_pos

,0,1,2,3,4
0,bc1_173-bc2_257-bc3_348,0,0,0,10
1,bc1_228-bc2_93-bc3_359,0,0,0,11
2,bc1_240-bc2_196-bc3_287,0,0,0,12
3,bc1_21-bc2_16-bc3_245,0,0,0,13
4,bc1_91-bc2_228-bc3_96,0,0,0,15
...,...,...,...,...,...
1844228,bc1_276-bc2_98-bc3_78,45,45,34,7
1844229,bc1_221-bc2_88-bc3_162,45,45,34,8
1844230,bc1_116-bc2_324-bc3_93,45,45,34,14
1844231,bc1_265-bc2_325-bc3_71,45,45,34,18


In [37]:
# Total number of spots in capture area
print(f"Total number of spots in capture area: {barcode_pos[0].nunique():,}")

Total number of spots in capture area: 1,844,233


In [38]:
# Total number of spots in table
n_spots = len(sdata.tables[f"{dataset_id}_level_1_table"].obs)
print(f"Number of spots in table: {n_spots:,}")

Number of spots in table: 1,844,233


In [39]:
# Number of spots in tissue (in table)
n_spots_in_tissue = len(sdata.tables[f"{dataset_id}_level_1_table"][sdata.tables[f"{dataset_id}_level_1_table"].obs['in_tissue_annotation'] == 1].obs)
print(f"Number of spots in tissue: {n_spots_in_tissue:,}")

Number of spots in tissue: 948,861


In [40]:
# Save barcodes
with open(os.path.join(os.path.dirname(output_path), "all_barcodes.txt"), "w") as f:
    for item in sdata.tables[f"{dataset_id}_level_1_table"].obs['barcode_name']:
        f.write(f"{item}\n")
        
with open(os.path.join(os.path.dirname(output_path), "barcodes_in_tissue.txt"), "w") as f:
    for item in sdata.tables[f"{dataset_id}_level_1_table"][sdata.tables[f"{dataset_id}_level_1_table"].obs['in_tissue_annotation'] == 1].obs['barcode_name']:
        f.write(f"{item}\n")      

In [41]:
# Area of tissue annotation that overlaps with capture area in mm²
tissue_gdf = sdata.shapes[f"{dataset_id}_tissue"].unary_union
capture_gdf = sdata.shapes[f"{dataset_id}_capture_area"].unary_union

overlap_geom = tissue_gdf.intersection(capture_gdf)

overlap_area_mm2 = (overlap_geom.area * microns_per_pixel**2) / 1000000

print(f"Capture area covered by tissue: {overlap_area_mm2:.2f} mm²")

Capture area covered by tissue: 21.80 mm²


## Capture effeciency metrics

In [42]:
gc.collect()

161

In [43]:
def row_sums(X):
    """Return 1D array of row sums without densifying."""
    return np.asarray(X.sum(axis=1)).ravel()

def col_sums(X):
    """Return 1D array of column sums without densifying."""
    return np.asarray(X.sum(axis=0)).ravel()

In [44]:
# Define Anndata
adata = sdata.tables[f"{dataset_id}_level_1_table"]

In [45]:
# Total number of spots in table with at least one UMI
n_spots_not_zero = (row_sums(adata.X) > 0).sum()
print(f"Number of spots in table with at least one UMI: {n_spots_not_zero:,}")

Number of spots in table with at least one UMI: 1,799,204


In [46]:
# Number of spots in tissue with at least one UMI
in_tissue_mask = adata.obs['in_tissue_annotation'] == 1
umis_mask = row_sums(adata.X) > 0
n_spots_in_tissue_not_zero = (in_tissue_mask & umis_mask).sum()
print(f"Number of spots in tissue with at least one UMI: {n_spots_in_tissue_not_zero:,}")

Number of spots in tissue with at least one UMI: 947,289


In [47]:
# Total number of genes in table
print(f"Total number of genes in table: {(col_sums(adata.X) > 0).sum():,}")

Total number of genes in table: 25,519


In [48]:
# Number of genes in tissue
print(f"Number of genes in tissue: {(col_sums(adata[adata.obs['in_tissue_annotation'] == 1].X) > 0).sum():,}")

Number of genes in tissue: 25,244


In [49]:
# Total number of unique UMIs
print(f"Total number of unique UMIs: {adata.X.sum():,.0f}")

Total number of unique UMIs: 42,612,604


In [50]:
# Number of unique UMIs in tissue
print(f"Number of unique UMIs in tissue: {adata[adata.obs['in_tissue_annotation'] == 1].X.sum():,.0f}")

Number of unique UMIs in tissue: 35,444,348


In [51]:
umi_gene = pd.read_csv(os.path.join(path, config_prefix, 'umi_gene.tsv.gz'), sep='\t', header=None, names=["barcode", "umi", "gene_id", "read_count"])
umi_gene

,barcode,umi,gene_id,read_count
0,bc1_10-bc2_380-bc3_369,GTTCCTACCAGC,ENSMUSG00000031647,2
1,bc1_10-bc2_380-bc3_369,AGTGTCTGTGGT,ENSMUSG00000024735,1
2,bc1_10-bc2_380-bc3_369,AAGCTAGTGCAA,ENSMUSG00000009995,6
3,bc1_10-bc2_380-bc3_369,AGAAGGGAAGCG,ENSMUSG00000021546,2
4,bc1_10-bc2_380-bc3_369,ATCGTATGGGCC,ENSMUSG00000006423,2
...,...,...,...,...
42612599,bc1_201-bc2_241-bc3_272,TAGTTCGGGG,ENSMUSG00000064370,1
42612600,bc1_201-bc2_241-bc3_272,TCTTACCTGC,ENSMUSG00000064354,1
42612601,bc1_201-bc2_241-bc3_272,ACCCCTAACC,ENSMUSG00000025716,1
42612602,bc1_200-bc2_242-bc3_115,AAGTTGGTACT,ENSMUSG00000030683,1


In [52]:
# Total number of UMIs
print(f"Total number of UMIs: {umi_gene['read_count'].sum():,}")

# Total number of unique UMIs
unique_UMIs = umi_gene[['barcode', 'umi', 'gene_id']].drop_duplicates().shape[0]
print(f"Total number of unique UMIs: {unique_UMIs:,}")

# Total number of unique barcodes
print(f"Total number of unique barcodes: {umi_gene['barcode'].nunique():,}")

# Total number of unique gene IDs
print(f"Total number of unique gene IDs: {umi_gene['gene_id'].nunique():,}")

Total number of UMIs: 76,172,912
Total number of unique UMIs: 42,612,604
Total number of unique barcodes: 1,799,204
Total number of unique gene IDs: 25,519


In [53]:
features = pd.read_csv(os.path.join(path, config_prefix, 'features.tsv'), sep='\t', header=None, names=["gene_id", "gene", "type"])
features

,gene_id,gene,type
0,ENSMUSG00000051951,Xkr4,Gene Expression
1,ENSMUSG00000089699,Gm1992,Gene Expression
2,ENSMUSG00000102331,Gm19938,Gene Expression
3,ENSMUSG00000102343,Gm37381,Gene Expression
4,ENSMUSG00000025900,Rp1,Gene Expression
...,...,...,...
32280,ENSMUSG00000095523,AC124606.1,Gene Expression
32281,ENSMUSG00000095475,AC133095.2,Gene Expression
32282,ENSMUSG00000094855,AC133095.1,Gene Expression
32283,ENSMUSG00000095019,AC234645.1,Gene Expression


In [54]:
print(f"Number of unique gene IDs: {features['gene_id'].nunique():,}")
print(f"Number of unique gene names: {features['gene'].nunique():,}")

Number of unique gene IDs: 32,285
Number of unique gene names: 32,245
